### Silver to Gold: Building BI Ready tables

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,IntegerType,DateType,TimestampType,FloatType

In [0]:
catalog_name='ecommerce'

In [0]:
df = spark.table(f"{catalog_name}.silver.slv_order_items")
df.limit(10).display()

In [0]:
# Add gross amount
df = df.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
)
# Add discount amount(discount_pct is already numeric, e.g., 21->21%)
df = df.withColumn(
    "discount_amount",
    F.ceil(F.col('gross_amount') * F.col('discount_pct')/100.0)
)
 # Add sale_amount = gross- amount
df = df.withColumn(
     'sale_amount',
     F.col("gross_amount")-F.col('discount_amount')+F.col('tax_amount')
)
 
# Add date id
df = df.withColumn(
    'date_id',
    F.date_format(F.col('dt'),'yyyyMMdd').cast(IntegerType()))
# coupon flag-coupon flag =1 if coupon code is not null else 0
df =df.withColumn(
    'coupon_flag',
    F.when(F.col("coupon_code").isNotNull(),F.lit(1))
    .otherwise(F.lit(0))
)    
df.limit(10).display()

In [0]:
# Currency Converstion
# 1) Define your fixed FX rates(as of 2026-11-16, like your PBI note)
fx_rates = {
"INR":     1.00,
"AED":     25.62,
"AUD":     63.40,
"CAD":     65.70,
"GBP":     123.94,
"SGD":     72.88,
"USD":     93.93,
}

rates = [(k,float(v)) for k,v in fx_rates.items() ]
rates_df = spark.createDataFrame(rates,["currency","inr_rate"])
rates_df.show()


In [0]:
df =(df.join(rates_df,rates_df.currency == F.upper(F.trim(F.col("unit_price_currency"))),
"left"
)
.withColumn("sale_amount_inr",F.col('sale_amount')* F.col('inr_rate'))
.withColumn("sale_amount_inr",F.ceil(F.col('sale_amount_inr')))
)

In [0]:
df.limit(10).display()







In [0]:
orders_gold_df = df.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")

)

In [0]:
orders_gold_df.limit(5).display()

In [0]:
# write raw data to gold layer(catalog:gold,schema:gold,table:gld_fact_order_items)
orders_gold_df.write.format("delta")\
.mode("overwrite")\
.option("mergeSchema",'true')\
.saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")            

In [0]:
# sanity check
spark.sql(f" select count(*) from {catalog_name}.gold.gld_fact_order_items").show()